# Starter Notebook: Qwen LoRA for Text-to-SVG (Kaggle + macOS)

This starter is built from the resources in `contest_docs`:
- Data resources: `contest_docs/03_Data_Design.md`
- Baseline and starter guidance: `contest_docs/05_Baselines_and_Starter_Notebooks.md`
- Kaggle implementation notes: `contest_docs/06_Kaggle_Implementation_Guide.md`

Goal: provide a practical scaffold for Qwen-class fine-tuning + submission generation on either CUDA/Linux or a local macOS machine.

The notebook now auto-selects a CUDA + Unsloth path when available and falls back to a standard Transformers + PEFT path on macOS.

## Referenced Data and Docs

### Dataset resources from `contest_docs/03_Data_Design.md`
- `OmniSVG/MMSVG-Icon`
- `xingxm/SVGX-Core-250k`
- `xingxm/SVGX-SFT-1M` (recommended subset: `SVGX_SFT_GEN_51k.json`)
- `nyuuzyou/svgfind`
- `starvector/svg-icons`
- `thesantatitan/deepseek-svg-dataset`
- `InternSVG/SArena` (evaluation benchmark)

### Qwen 2B fine-tuning references from `contest_docs/05` and `contest_docs/06`
- Unsloth Qwen fine-tune docs: https://unsloth.ai/docs/models/qwen3.5/fine-tune
- Qwen3.5-2B Vision notebook: https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Qwen3_5_(2B)_Vision.ipynb

Note: this notebook is written as a reusable starter. You may need to adjust exact model IDs and column names to match the latest upstream datasets.

In [11]:
# Install dependencies in a fresh environment.
# CUDA/Linux:
# %pip install -q unsloth datasets trl transformers accelerate peft bitsandbytes pandas lxml
#
# macOS:
%pip install -q datasets trl transformers accelerate peft pandas lxml sentencepiece tiktoken

Note: you may need to restart the kernel to use updated packages.


In [12]:
import os
import re
import time
import random
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import torch

from datasets import Dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

Torch: 2.10.0
CUDA available: False


In [13]:
# Core training config.
from pathlib import Path
import platform


def detect_backend():
    if torch.cuda.is_available():
        return "cuda"

    mps_backend = getattr(torch.backends, "mps", None)
    if mps_backend and mps_backend.is_available():
        return "mps"

    return "cpu"


BACKEND = detect_backend()
DEVICE = torch.device(BACKEND)
USE_UNSLOTH = BACKEND == "cuda"

local_model_dir = Path(os.environ.get("QWEN_MODEL_DIR", "./models/qwen2.5-3B-Instruct"))

default_output_dir = (
    "/kaggle/working/qwen3b_svg_lora_v2"
    if Path("/kaggle/working").exists()
    else str(Path("./artifacts/qwen3b_svg_lora_v2").resolve())
)

if USE_UNSLOTH:
    default_model_name = "unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit"
elif local_model_dir.exists():
    default_model_name = str(local_model_dir.resolve())
else:
    default_model_name = "Qwen/Qwen2.5-3B-Instruct"

CONFIG = {
    "model_name": default_model_name,
    "max_seq_length": 1024,
    "lora_r": 16,
    "lora_alpha": 32,
    "learning_rate": 2e-4,
    "num_train_epochs": 2,
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 16 if USE_UNSLOTH else 8,
    "warmup_ratio": 0.05,
    "weight_decay": 0.01,
    "logging_steps": 20,
    "eval_steps": 100,
    "save_steps": 200,
    "max_train_samples_per_source": 2000,
    "eval_size": 0.02,
    "output_dir": default_output_dir,
}

print(f"Platform: {platform.system()} {platform.machine()}")
print(f"Backend: {BACKEND}")
if BACKEND == "cpu":
    print("Warning: neither CUDA nor MPS is available, so training will run on CPU.")
print(f"Using model: {CONFIG['model_name']}")
print(f"Using output_dir: {CONFIG['output_dir']}")
CONFIG

Platform: Darwin arm64
Backend: mps
Using model: Qwen/Qwen2.5-3B-Instruct
Using output_dir: /Users/bolajialabi/Projects/Deep_Learning_Midterm_1/artifacts/qwen3b_svg_lora_v2


{'model_name': 'Qwen/Qwen2.5-3B-Instruct',
 'max_seq_length': 1024,
 'lora_r': 16,
 'lora_alpha': 32,
 'learning_rate': 0.0002,
 'num_train_epochs': 2,
 'per_device_train_batch_size': 1,
 'gradient_accumulation_steps': 8,
 'warmup_ratio': 0.05,
 'weight_decay': 0.01,
 'logging_steps': 20,
 'eval_steps': 100,
 'save_steps': 200,
 'max_train_samples_per_source': 2000,
 'eval_size': 0.02,
 'output_dir': '/Users/bolajialabi/Projects/Deep_Learning_Midterm_1/artifacts/qwen3b_svg_lora_v2'}

In [14]:
# Competition training source (VS Code + Colab extension friendly)
from pathlib import Path

IS_COLAB = False
try:
    from google.colab import drive
    IS_COLAB = True
    drive.mount('/content/drive', force_remount=False)
except ImportError:
    pass

if IS_COLAB:
    DATA_DIR = Path('/content/drive/MyDrive/DL_M1')
    TRAIN_CSV_PATH = str(DATA_DIR / 'train.csv')
    TEST_CSV_PATH = str(DATA_DIR / 'test.csv')
else:
    TRAIN_CSV_PATH = 'data/train.csv'
    TEST_CSV_PATH = 'data/test.csv'

if not Path(TRAIN_CSV_PATH).exists():
    raise FileNotFoundError(f'Train file not found: {TRAIN_CSV_PATH}')
if not Path(TEST_CSV_PATH).exists():
    raise FileNotFoundError(f'Test file not found: {TEST_CSV_PATH}')

print(f'Using train file: {TRAIN_CSV_PATH}')
print(f'Using test file: {TEST_CSV_PATH}')

Using train file: data/train.csv
Using test file: data/test.csv


In [15]:
train_df = pd.read_csv(TRAIN_CSV_PATH)
required_cols = {"prompt", "svg"}
missing = required_cols - set(train_df.columns)
if missing:
    raise ValueError(f"Missing required columns in {TRAIN_CSV_PATH}: {sorted(missing)}")

train_df = train_df.dropna(subset=["prompt", "svg"]).copy()
train_df["prompt"] = train_df["prompt"].astype(str).str.strip()
train_df["svg"] = train_df["svg"].astype(str).str.strip()
train_df = train_df[(train_df["prompt"] != "") & (train_df["svg"].str.lower().str.startswith("<svg"))]

if CONFIG["max_train_samples_per_source"] and len(train_df) > CONFIG["max_train_samples_per_source"]:
    train_df = train_df.sample(CONFIG["max_train_samples_per_source"], random_state=SEED)

if len(train_df) == 0:
    raise RuntimeError("No usable rows found in train.csv after filtering.")

train_raw = Dataset.from_pandas(train_df[["prompt", "svg"]], preserve_index=False)
train_raw = train_raw.shuffle(seed=SEED)

splits = train_raw.train_test_split(test_size=CONFIG["eval_size"], seed=SEED)
train_ds = splits["train"]
eval_ds = splits["test"]

print(f"Train rows: {len(train_ds)}")
print(f"Eval rows: {len(eval_ds)}")
train_ds[0]

Train rows: 1960
Eval rows: 40


{'prompt': 'A black abstract symbol consisting of three circular elements connected by curved lines, set against a white background. The top circle is positioned slightly left, with two larger circles below itâ\x80\x94one to the left and one to the rightâ\x80\x94connected by smooth, flowing black lines forming an H-like structure.',
 'svg': '<svg xmlns="http://www.w3.org/2000/svg" viewBox="0.0 0.0 200.0 200.0" height="200.0px" width="200.0px"><path fill="#000000" fill-opacity="1.0"  filling="0" d="M59.20800018310547 73.25 A25.007999420166016 25.007999420166016 0.0 0 0 83.33300018310547 91.66699981689453 L116.66699981689453 91.66699981689453 A41.68299865722656 41.68299865722656 0.0 0 1 157.72500610351562 126.21700286865234 A25.007999420166016 25.007999420166016 0.0 0 1 150.0 175.0 A25.0 25.0 0.0 0 1 140.79200744628906 126.75 A25.007999420166016 25.007999420166016 0.0 0 0 116.66699981689453 108.33300018310547 L83.33300018310547 108.33300018310547 A41.483001708984375 41.483001708984375 0.

In [16]:
SYSTEM_PROMPT = (
    "You generate compact, valid SVG markup from user requests. "
    "Return only SVG code with a single root <svg> element."
)


def format_sft_text(example):
    text = (
        "<|im_start|>system\n"
        f"{SYSTEM_PROMPT}<|im_end|>\n"
        "<|im_start|>user\n"
        f"{example['prompt']}<|im_end|>\n"
        "<|im_start|>assistant\n"
        f"{example['svg']}<|im_end|>"
    )
    return {"text": text}


train_text = train_ds.map(format_sft_text, remove_columns=train_ds.column_names)
eval_text = eval_ds.map(format_sft_text, remove_columns=eval_ds.column_names)

print(train_text[0]["text"][:400])

Map: 100%|██████████| 40/40 [00:00<00:00, 7090.66 examples/s]

<|im_start|>system
You generate compact, valid SVG markup from user requests. Return only SVG code with a single root <svg> element.<|im_end|>
<|im_start|>user
A black abstract symbol consisting of three circular elements connected by curved lines, set against a white background. The top circle is positioned slightly left, with two larger circles below itâone to the left and one to the rightâc


In [17]:
try:
    adapter_path = Path(CONFIG["output_dir"])
    saved_adapter_exists = adapter_path.exists() and (adapter_path / "adapter_model.safetensors").exists()
    tokenizer_source = str(adapter_path) if (adapter_path / "tokenizer.json").exists() else CONFIG["model_name"]

    if USE_UNSLOTH:
        from peft import PeftModel
        from unsloth import FastLanguageModel

        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=CONFIG["model_name"],
            max_seq_length=CONFIG["max_seq_length"],
            dtype=None,
            load_in_4bit=True,
        )

        if saved_adapter_exists:
            model = PeftModel.from_pretrained(model, str(adapter_path), is_trainable=True)
        else:
            model = FastLanguageModel.get_peft_model(
                model,
                r=CONFIG["lora_r"],
                lora_alpha=CONFIG["lora_alpha"],
                lora_dropout=0,
                bias="none",
                target_modules=[
                    "q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj",
                ],
                use_gradient_checkpointing="unsloth",
                random_state=SEED,
            )
    else:
        from peft import LoraConfig, PeftModel, get_peft_model
        from transformers import AutoModelForCausalLM, AutoTokenizer

        tokenizer = AutoTokenizer.from_pretrained(tokenizer_source)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        tokenizer.padding_side = "right"

        model = AutoModelForCausalLM.from_pretrained(
            CONFIG["model_name"],
            torch_dtype=torch.float16 if BACKEND != "cpu" else torch.float32,
            low_cpu_mem_usage=True,
        )
        if saved_adapter_exists:
            model = PeftModel.from_pretrained(model, str(adapter_path), is_trainable=True)
        else:
            model = get_peft_model(
                model,
                LoraConfig(
                    r=CONFIG["lora_r"],
                    lora_alpha=CONFIG["lora_alpha"],
                    lora_dropout=0.0,
                    bias="none",
                    task_type="CAUSAL_LM",
                    target_modules=[
                        "q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj",
                    ],
                ),
            )
        if BACKEND != "cpu":
            model = model.to(DEVICE)
except OSError as exc:
    raise RuntimeError(
        "Model download/load failed. If you are on macOS, either give the notebook network access to Hugging Face "
        "or set QWEN_MODEL_DIR (or ./models/qwen2.5-3B-Instruct) to a local downloaded model directory. "
        f"Current model setting: {CONFIG['model_name']} | Adapter dir: {CONFIG['output_dir']}"
    ) from exc

model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False
MODEL_DEVICE = next(model.parameters()).device

print(f"Loaded model on: {MODEL_DEVICE}")
if hasattr(model, "print_trainable_parameters"):
    model.print_trainable_parameters()

Loading weights: 100%|██████████| 434/434 [00:10<00:00, 42.41it/s]


Loaded model on: mps:0
trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


In [13]:
from trl import SFTConfig, SFTTrainer

precision_kwargs = {"fp16": False, "bf16": False}
if BACKEND == "cuda":
    precision_kwargs["bf16"] = torch.cuda.is_bf16_supported()
    precision_kwargs["fp16"] = not precision_kwargs["bf16"]

training_args = SFTConfig(
    output_dir=CONFIG["output_dir"],
    num_train_epochs=CONFIG["num_train_epochs"],
    per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
    per_device_eval_batch_size=CONFIG["per_device_train_batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    learning_rate=CONFIG["learning_rate"],
    warmup_ratio=CONFIG["warmup_ratio"],
    weight_decay=CONFIG["weight_decay"],
    logging_steps=CONFIG["logging_steps"],
    eval_strategy="steps",
    eval_steps=CONFIG["eval_steps"],
    save_strategy="steps",
    save_steps=CONFIG["save_steps"],
    save_total_limit=3,
    save_only_model=False,
    report_to="none",
    optim="paged_adamw_8bit" if USE_UNSLOTH else "adamw_torch",
    lr_scheduler_type="cosine",
    seed=SEED,
    dataset_text_field="text",
    max_length=CONFIG["max_seq_length"],
    packing=True,
    gradient_checkpointing=USE_UNSLOTH,
    use_cpu=BACKEND == "cpu",
    dataloader_pin_memory=BACKEND == "cuda",
    **precision_kwargs,
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_text,
    eval_dataset=eval_text,
    args=training_args,
)

resume_ckpt = None
output_path = Path(CONFIG["output_dir"])
if output_path.exists():
    ckpts = sorted(output_path.glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[1]))
    for ckpt in reversed(ckpts):
        if (ckpt / "adapter_model.safetensors").exists() or (ckpt / "model.safetensors").exists():
            resume_ckpt = str(ckpt)
            break
if resume_ckpt:
    print(f"Resuming training from: {resume_ckpt}")
else:
    print("Starting training from scratch.")

train_result = trainer.train(resume_from_checkpoint=resume_ckpt)
train_result

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[RANK 0] Padding-free training is enabled, but the attention implementation is not set to a supported flash attention variant. Padding-free training flattens batches into a single sequence, and only the following implementations are known to reliably support this: flash_attention_2, flash_attention_3, kernels-community/flash-attn, kernels-community/flash-attn3, kernels-community/vllm-flash-attn3. Using other implementations may lead to unexpected behavior. To ensure compatibility, set `attn_implementation` in the model configuration to one of these supported options or verify that your attention mechanism can handle flattened sequences.
[RANK 0] You are using packing, but the attention implementation is not set to a supported flash attention variant. Packing gathers multiple samples into a single sequence, and only the following implementations are known to reliably support this: flash_attention_2, flas

Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
100,0.645714,0.542897,0.545860,813656.000000,0.821464
200,0.572354,0.478934,0.484881,1627439.000000,0.841779
300,0.526248,0.456737,0.443736,2436917.000000,0.849272
400,0.479666,0.450077,0.446649,3250651.000000,0.851720


TrainOutput(global_step=440, training_loss=0.5848976005207408, metrics={'train_runtime': 51440.83, 'train_samples_per_second': 0.068, 'train_steps_per_second': 0.009, 'total_flos': 7858731905687040.0, 'train_loss': 0.5848976005207408, 'epoch': 2.0})

In [10]:
output_dir = Path(CONFIG["output_dir"])
output_dir.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(output_dir))
tokenizer.save_pretrained(str(output_dir))

print(f"Saved adapter + tokenizer to: {output_dir}")

NameError: name 'trainer' is not defined

In [18]:
from peft import PeftModel

adapter_path = Path(CONFIG["output_dir"])
if adapter_path.exists() and (adapter_path / "adapter_model.safetensors").exists():
    if hasattr(model, "peft_config"):
        print(f"LoRA adapter already attached from: {adapter_path}")
    else:
        model = PeftModel.from_pretrained(model, str(adapter_path), is_trainable=True)
        if BACKEND != "cpu":
            model = model.to(DEVICE)
        print(f"Loaded LoRA adapter from: {adapter_path}")
else:
    print("No saved adapter found, using model from training.")

LoRA adapter already attached from: /Users/bolajialabi/Projects/Deep_Learning_Midterm_1/artifacts/qwen3b_svg_lora_v2


In [19]:
if USE_UNSLOTH:
    FastLanguageModel.for_inference(model)

tokenizer.padding_side = "left"
model.config.use_cache = True
model.eval()
MODEL_DEVICE = next(model.parameters()).device

SVG_REGEX = re.compile(r"<svg.*?</svg>", flags=re.IGNORECASE | re.DOTALL)


def extract_svg(text):
    m = SVG_REGEX.search(text)
    return m.group(0).strip() if m else ""


def is_valid_svg(svg_text):
    if not svg_text:
        return False
    try:
        root = ET.fromstring(svg_text)
        return root.tag.endswith("svg")
    except ET.ParseError:
        return False


def fallback_svg(_prompt):
    return (
        '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256">'
        '<rect x="0" y="0" width="256" height="256" fill="white"/>'
        '<circle cx="128" cy="128" r="64" fill="black"/>'
        '</svg>'
    )


def generate_svg(prompt, max_new_tokens=512):
    chat_text = (
        "<|im_start|>system\n"
        f"{SYSTEM_PROMPT}<|im_end|>\n"
        "<|im_start|>user\n"
        f"{prompt}<|im_end|>\n"
        "<|im_start|>assistant\n"
    )
    inputs = tokenizer(chat_text, return_tensors="pt").to(MODEL_DEVICE)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.02,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    decoded = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    svg = extract_svg(decoded)
    if not is_valid_svg(svg):
        svg = fallback_svg(prompt)
    return svg


def generate_svg_batch(prompts, max_new_tokens=512):
    chat_texts = [
        (
            "<|im_start|>system\n"
            f"{SYSTEM_PROMPT}<|im_end|>\n"
            "<|im_start|>user\n"
            f"{prompt}<|im_end|>\n"
            "<|im_start|>assistant\n"
        )
        for prompt in prompts
    ]
    inputs = tokenizer(
        chat_texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=CONFIG["max_seq_length"],
    ).to(MODEL_DEVICE)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.02,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    decoded_batch = tokenizer.batch_decode(output_ids, skip_special_tokens=True)
    svgs = []
    for prompt, decoded in zip(prompts, decoded_batch):
        svg = extract_svg(decoded)
        if not is_valid_svg(svg):
            svg = fallback_svg(prompt)
        svgs.append(svg)
    return svgs


test_prompt = "a simple blue bird icon"
pred_svg = generate_svg(test_prompt)
print(pred_svg[:500])
print("Valid SVG:", is_valid_svg(pred_svg))

<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256"><rect x="0" y="0" width="256" height="256" fill="white"/><circle cx="128" cy="128" r="64" fill="black"/></svg>
Valid SVG: True


In [ ]:
# Submission generation scaffold: expects Kaggle prompt file with columns `id,prompt`.
# TEST_PROMPTS_PATH = "/kaggle/input/svg-test-public-prompts/test_prompts.csv"
# SUBMISSION_PATH = "/kaggle/working/submission.csv"

# Save locally
TEST_PROMPTS_PATH = TEST_CSV_PATH  
SUBMISSION_PATH = "./submission.csv"  

test_df = pd.read_csv(TEST_PROMPTS_PATH)

batch_size = 16
rows = []
invalid_count = 0
t0 = time.time()

for start in range(0, len(test_df), batch_size):
    batch_df = test_df.iloc[start:start + batch_size]
    prompts = batch_df["prompt"].tolist()
    ids = batch_df["id"].tolist()

    svgs = generate_svg_batch(prompts, max_new_tokens=512)

    for sample_id, prompt, svg in zip(ids, prompts, svgs):
        if not is_valid_svg(svg):
            invalid_count += 1
            svg = fallback_svg(prompt)
        rows.append({"id": sample_id, "svg": svg}) 
elapsed_min = (time.time() - t0) / 60
print(f"Saved: {SUBMISSION_PATH}")
print(f"Rows: {len(sub_df)}")
print(f"Invalid/fallback count: {invalid_count}")
print(f"Runtime (minutes): {elapsed_min:.2f}")
sub_df.head()

## Notes

- Keep a fixed seed, runtime logs, and invalid-generation counts (required by `contest_docs/05`).
- If you use Kaggle-packaged datasets (`svg-train-public`, `svg-test-public-prompts`, `svg-utils`), swap paths into the loading cells.
- On macOS, the notebook falls back to `Qwen/Qwen2.5-0.5B-Instruct` with standard Transformers + PEFT because Unsloth and bitsandbytes are CUDA-only.
- If `mps` is unavailable, the notebook will still run on CPU, but training will be much slower.
- The first local run must be able to download the model from Hugging Face unless you point `QWEN_MODEL_DIR` (or `./models/qwen2.5-0.5B-Instruct`) at a local copy.
- For stricter alignment with Unsloth templates, copy the latest prompt-formatting snippets from the official Qwen3.5-2B notebook linked above.